# Gradio Application

## Set-Up

In [1]:
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("application").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/22 17:28:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/22 17:29:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel, GBTClassificationModel

# TODO: Get pipelines to feed data through
tabular_pipeline = PipelineModel.load("hdfs://localhost:9000/user/jj/final_proj/tabular_pipeline.parquet")
lr_model = LogisticRegressionModel.load("hdfs://localhost:9000/user/jj/final_proj/lr_model")
rf_model = RandomForestClassificationModel.load("hdfs://localhost:9000/user/jj/final_proj/rf_model")
gbt_model = GBTClassificationModel.load("hdfs://localhost:9000/user/jj/final_proj/gbt_model")

## Creating Prediction Methods

In [17]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

threshold = 0.91

extract_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())


def predict_from_tabular(age, bmi, alcohol, gender, smoker, chf, chd, heart_attack, stroke, thyroid, race):
    # Convert inputs into pandas DataFrame
    df = pd.DataFrame([{
        "Age": age,
        "BMI": bmi,
        "Alcohol/Day": alcohol,
        "Gender": 1 if gender == "Male" else 0,
        "Smokes?": 1 if smoker == "Yes" else 0,
        "Congestive Heart Failure": 1 if chf == "Yes" else 0,
        "Coronary Heart Disease": 1 if chd == "Yes" else 0,
        "Heart Attack": 1 if heart_attack == "Yes" else 0,
        "Stroke": 1 if stroke == "Yes" else 0,
        "Thyroid": 1 if thyroid == "Yes" else 0,
        "Race": race
    }])

    # Convert to Spark DataFrame
    sdf = spark.createDataFrame(df)

    # Apply preprocessing
    prepped = tabular_pipeline.transform(sdf)

    # Get model predictions
    rf_preds = rf_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("rf_prob"))
    gbt_preds = gbt_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("gbt_prob"))
    lr_preds = lr_model.transform(prepped).select(extract_prob_1(F.col("probability")).alias("lr_prob"))

    # Add row IDs and join
    prepped = prepped.withColumn("row_id", F.monotonically_increasing_id())
    rf_preds = rf_preds.withColumn("row_id", F.monotonically_increasing_id())
    gbt_preds = gbt_preds.withColumn("row_id", F.monotonically_increasing_id())
    lr_preds = lr_preds.withColumn("row_id", F.monotonically_increasing_id())

    ensemble_df = prepped \
        .join(rf_preds, "row_id") \
        .join(gbt_preds, "row_id") \
        .join(lr_preds, "row_id") \
        .withColumn("avg_prob", (F.col("rf_prob") + F.col("gbt_prob") + F.col("lr_prob")) / 3) \
        .withColumn("ensemble_prediction", F.when(F.col("avg_prob") > threshold, 1).otherwise(0).cast("double"))

    result = ensemble_df.select("ensemble_prediction", "avg_prob").collect()[0]
    label = "Likely has cancer" if result["ensemble_prediction"] == 1 else "Unlikely to have cancer"
    confidence = result["avg_prob"]

    # return (label, confidence)
    return f"{label} (Confidence: {confidence:.2f})"



In [12]:
def predict_from_image(image):
    # Preprocess and predict using MRI model
    # e.g., PyTorch/TensorFlow logic here
    # prediction = mri_model.predict(image)  # placeholder
    # return float(prediction)
    return None

In [5]:
# TODO: Combined prediction
def dual_prediction(image, age, bmi, alcohol, gender, smoker, chf, chd, heart_attack, stroke, thyroid, race):
    image_pred = predict_from_image(image)
    tabular_pred = predict_from_tabular(age, bmi, alcohol, gender, smoker, chf, chd, heart_attack, stroke, thyroid, race)

    # TODO: Figure out how to combine the two predictions and return
    # return (image_pred, tabular_pred)
    

In [18]:
import gradio as gr

iface = gr.Interface(
    fn=predict_from_tabular,
    # fn=dual_prediction,
    inputs=[
        # gr.Image(type="numpy", label="MRI Scan"),
        gr.Slider(18, 100, step=1, label="Age"),
        gr.Slider(10.0, 60.0, step=0.1, label="BMI"),
        gr.Slider(0.0, 10.0, step=0.1, label="Alcohol/Day"),
        gr.Radio(["Male", "Female"], label="Gender"),
        gr.Radio(["Yes", "No"], label="Smokes?"),
        gr.Radio(["Yes", "No"], label="Congestive Heart Failure"),
        gr.Radio(["Yes", "No"], label="Coronary Heart Disease"),
        gr.Radio(["Yes", "No"], label="Heart Attack"),
        gr.Radio(["Yes", "No"], label="Stroke"),
        gr.Radio(["Yes", "No"], label="Thyroid"),
        gr.Radio(["White", "Black", "Hispanic", "Asian", "Other"], label="Race")
        
    ],
    outputs="text",
    title="Cancer Risk Predictor",
    description="Enter patient data to predict likelihood of cancer using a Spark-based ensemble model."
)

iface.launch(share=True)


Running on local URL:  http://127.0.0.1:7866
Running on public URL: https://afa0417ae63e5b7d06.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


25/04/22 18:33:27 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
25/04/22 18:33:28 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/22 18:33:33 ERROR Executor: Exception in task 1.0 in stage 57.0 (TID 59)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`StringIndexerModel$$Lambda/0x00007fde390c0000`: (string) => double).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:198)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(Whole